# MCS Selection - Final Analysis Notebook

This notebook loads trained models and generates final analysis plots for the project.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Import project modules
sys.path.append(str(Path('.').resolve()))
from training.config import set_global_seed, GLOBAL_SEED
from models.baselines import get_model
from evaluation.visualization import set_plot_style, plot_confusion_matrix, plot_attention_heatmaps
from evaluation.metrics import load_metrics_file

set_global_seed(GLOBAL_SEED)
set_plot_style()
print('Project loaded successfully!')

## Load All Model Metrics

In [ ]:
from evaluation.compare_baselines import load_all_model_metrics

model_names = ['attention', 'dnn', 'cnn', 'lstm', 'cnn_lstm']
all_metrics = load_all_model_metrics('results/', model_names)

print('Loaded metrics for models:', list(all_metrics.keys()))

# Create comparison table
for model, metrics in all_metrics.items():
    print(f"\n{model.upper()}:")
    print(f"  Accuracy: {metrics.get('test_accuracy', 0)*100:.2f}%")
    print(f"  Throughput: {metrics.get('throughput_ratio', 0):.4f}")
    print(f"  Latency: {metrics.get('inference_latency_ms', 'N/A')} ms")

## Per-Class Accuracy Bar Chart

In [ ]:
from evaluation.visualization import plot_class_accuracy_comparison

class_accuracies = {}
for model_name, metrics in all_metrics.items():
    if 'per_class_accuracy' in metrics:
        class_accuracies[model_name] = metrics['per_class_accuracy']

plot_class_accuracy_comparison(class_accuracies, save_path='results/plots/class_accuracy_comparison.png')

## Attention Visualization (for Attention Model Only)

In [ ]:
import torch
from training.data_generator import load_split

# Load a batch from test set
test_channels, test_labels = load_split('data/test')
test_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(torch.from_numpy(test_channels).float(), torch.from_numpy(test_labels).long()),
    batch_size=32, shuffle=False
)

sample_batch = next(iter(test_loader))
channels, labels = sample_batch

# Load attention model
from models.attention_mcs import AttentionMCSModel
from training.config import DataConfig, AttentionModelConfig

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint = torch.load('results/attention/checkpoints/best_model.pt', map_location=device)

attention_config = AttentionModelConfig()
model = AttentionMCSModel().to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Get attention weights for a few samples
with torch.no_grad():
    logits, attn_weights_list = model.get_attention_weights(channels.to(device))

print(f'Number of attention layers: {len(attn_weights_list)}')
print(f'Attention weights shape (layer 0): {attn_weights_list[0].shape}')  # [batch, n_heads, seq_len, seq_len]

# Plot attention for first sample in batch
plot_attention_heatmaps(attn_weights_list, sample_idx=0, n_heads_to_plot=4,
                       save_path='results/plots/attention_heatmaps_sample.png')

## Training Curves

In [ ]:
from evaluation.visualization import plot_training_history
import json

# Load training history
history_path = Path('results/attention/training_history.json')
if history_path.exists():
    with open(history_path, 'r') as f:
        history = json.load(f)
    plot_training_history(history, save_path='results/plots/attention_training_history.png')

## Final Results Table

In [ ]:
from evaluation.metrics import summarize_metrics_comparison

summary = summarize_metrics_comparison(all_metrics)
df_summary = pd.DataFrame(summary, index=model_names).T
print('\nFinal Results Summary:')
print(df_summary.round(4))

# Save as CSV
df_summary.to_csv('results/plots/comparison_summary.csv')
print('\nSummary saved to results/plots/comparison_summary.csv')